# Stage 3 & 4: Supervised Model Building, Evaluation & Executive Synthesis
### Campus Placement Classification (100,000 Records Benchmark)
**Assigned to:** Punit (Project Lead)  
**Project:** IT462 Campus Placement Prediction & Package Analytics (Weightage: 50%)

---
## 📌 Instructions for Punit:
Your objective is to build supervised classification pipelines to accurately predict campus placement, compare linear vs tree-based models, extract feature importance to answer whether CPI vs Coding/Projects dominates hiring decisions, and deliver executive-grade visualizations and metrics.

### Tasks to Complete:
- [ ] **1. Data Preparation:** Load encoded dataset (with fallback to raw data + auto-preprocessing).
- [ ] **2. Train-Test Split & Scaling:** Perform 80/20 Stratified Train-Test split and scale continuous variables with `StandardScaler`.
- [ ] **3. Model Training (4 Classifiers):**
  - Logistic Regression (Baseline linear benchmark)
  - Decision Tree Classifier (Interpretable decision rules)
  - Random Forest Classifier (Bagging ensemble)
  - Gradient Boosting / LightGBM (Boosting powerhouse)
- [ ] **4. Model Evaluation:** Compute Accuracy, Precision, Recall, F1-Score, and ROC-AUC for all 4 models.
- [ ] **5. Visual Evaluations:** Plot comparative ROC curves and Confusion Matrices.
- [ ] **6. Feature Importance:** Extract Gini importance from Random Forest/Gradient Boosting to answer the core research question: What drives placement most?
- [ ] **7. Model Comparison Table:** Summarize metric scores across all models in a clean Markdown/DataFrame table.
- [ ] **8. Executive Conclusions:** Highlight actionable findings for placement committees and students.

## 1. Import Libraries
> **TODO (Punit):** Import modeling modules, metrics, and visualization utilities.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

## 2. Load Dataset & Train-Test Split
> **TODO (Punit):** Load dataset, split features `X` and target `y` (target: `placement_status`), and create 80/20 stratified split.

In [ ]:
encoded_path = '../data/processed/placement_data_encoded.csv'
raw_path = '../data/raw/student_placement_data.csv'

if os.path.exists(encoded_path):
    df = pd.read_csv(encoded_path)
else:
    # Fallback auto-prep directly from raw benchmark
    df = pd.read_csv(raw_path)
    drop_cols = ['student_id', 'salary_lpa']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    if 'branch' in df.columns:
        df = pd.get_dummies(df, columns=['branch'], drop_first=True)

X = df.drop(columns=['placement_status'])
y = df['placement_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 3. Model Training & Evaluation Pipeline
> **TODO (Punit):** Train the 4 classifiers and compute metrics.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)
    })

df_results = pd.DataFrame(results)
df_results

## 4. Feature Importance Analysis
> **TODO (Punit):** Plot feature importances from Random Forest to see whether CPI, Coding, or Experience dominates.

In [ ]:
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance: Key Drivers of Campus Placement')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()